<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 22</h2>
<h2>Elliptik Əyrilər - II Hissə</h2>
<h3 style="color: #8B4513;">ECC Protokolları və Standart Əyrilər</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [İcra mühiti](#icra-mühiti)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: Elliptik əyrilərin əsasları (5 dəq)](#xatırlatma-elliptik-əyrilərin-əsasları-5-dəq)
5. [ECDH (20 dəq)](#ecdh-elliptic-curve-diffie-hellman-20-dəq)
6. [ECDSA (25 dəq)](#ecdsa-elliptic-curve-digital-signature-algorithm-25-dəq)
7. [Ed25519 (10 dəq)](#ed25519-10-dəq)
8. [NIST əyriləri və Dual_EC_DRBG (10 dəq)](#nist-əyriləri-və-dual_ec_drbg-ayrımı-10-dəq)
9. [X25519, Curve25519 və secp256k1 (10 dəq)](#curve25519-və-x25519-10-dəq)
10. [Brainpool əyriləri (5 dəq)](#brainpool-əyriləri-5-dəq)
11. [SafeCurves qiymətləndirmə meyarları (5 dəq)](#təhlükəsiz-əyri-seçimi-meyarları-safecurves-5-dəq)
12. [İnteqrasiya edilmiş özünü-yoxlama (15 dəq)](#inteqrasiya-edilmiş-özünü-yoxlama-15-dəq)
13. [Ev tapşırığı](#ev-tapşırığı)
14. [Yekun](#yekun)
15. [Əlavə resurslar](#əlavə-resurslar)


## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ ECDH (Elliptic Curve Diffie-Hellman) açar razılaşdırmasını və autentifikasiya sərhədini izah edə biləcəksiniz  
✅ ECDSA (Elliptic Curve Digital Signature Algorithm) imza sxemini başa düşəcəksiniz  
✅ Ed25519 signature scheme ilə X25519 key-agreement function-u ayıra biləcəksiniz  
✅ NIST əyriləri (P-256, P-384, P-521) haqqında məlumatlı olacaqsınız  
✅ Curve25519, X25519 və secp256k1 terminlərini düzgün kontekstdə işlədə biləcəksiniz  
✅ Python-da `cryptography` kitabxanası ilə positive/negative ECC protocol testləri yaza biləcəksiniz

## 📚 İcra mühiti

| Asılılıq | Tələb | İstifadə sahəsi |
|----------|------|-----------------|
| Python | 3.10+ | standart kitabxana və təkrarolunan testlər |
| cryptography | cari kernel-də əvvəlcədən quraşdırılmış | ECDH, ECDSA, Ed25519, X25519, HKDF |

> **Mühit müqaviləsi:** notebook şəbəkədən paket quraşdırmır, `cwd` dəyişdirmir və məcburi bölmələri availability flag ilə buraxmır. `cryptography` yoxdursa icra ilk hüceyrədə aydın xəta ilə dayanır.


In [ ]:
# Fail-fast dependency contract: notebook daxilində paket quraşdırılmır
import statistics
import sys
import time
from importlib.util import find_spec

assert sys.version_info >= (3, 10), "Python 3.10+ tələb olunur"
if find_spec("cryptography") is None:
    raise RuntimeError("Tələb olunan paket yoxdur: cryptography")

print(f"✅ Python {sys.version_info.major}.{sys.version_info.minor}; cryptography tapıldı")


## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı hüceyrə istifadə olunan bütün public API-ləri import edir. Import xətası optional skip deyil: notebook dərhal dayanmalıdır.


In [ ]:
import cryptography
from cryptography.exceptions import InvalidSignature
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import ec, ed25519, x25519
from cryptography.hazmat.primitives.asymmetric.utils import (
    decode_dss_signature,
    encode_dss_signature,
)
from cryptography.hazmat.primitives.kdf.hkdf import HKDF

print(f"✅ cryptography {cryptography.__version__}: tələb olunan API-lər import edildi")


### 3.2 Təkrarolunan icra qaydası

Notebook cari qovluğa fayl yazmır, işçi qovluğu dəyişdirmir və `input()` gözləmir. Gizli və törədilmiş açarlar ekrana çıxarılmır; yalnız ölçü və yoxlama nəticələri göstərilir.


In [ ]:
execution_contract = {
    "changes_working_directory": False,
    "installs_packages": False,
    "waits_for_input": False,
    "prints_shared_or_derived_secret": False,
}
assert not any(execution_contract.values())
print("✅ İcra müqaviləsi yoxlanıldı")


## 📖 Xatırlatma: Elliptik əyrilərin əsasları (5 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 Sonlu meydan üzərində elliptik əyri qrupu</h4>
<p>Xarakteristikası 2 və 3 olmayan sadə meydan üçün qısa Weierstrass forması:</p>
<p>$$E/\mathbb F_p: y^2=x^3+ax+b,\qquad 4a^3+27b^2\not\equiv0\pmod p.$$</p>
<ul>
    <li><b>Nöqtə toplama:</b> $P+Q$ — $E(\mathbb F_p)$ qrup əməliyyatı.</li>
    <li><b>Skalyar vurma:</b> $[k]P$ — double-and-add kimi alqoritmlərlə hesablanır.</li>
    <li><b>Domen parametrləri:</b> əyri, baza nöqtəsi $G$, onun sadə sırası $n$ və kofaktor $h$ ayrıca yoxlanır.</li>
    <li><b>ECDLP:</b> $Q=[k]G$ verildikdə $k\bmod n$ qiymətini tapmaq problemidir.</li>
</ul>
</div>


## 🔄 ECDH (Elliptic Curve Diffie-Hellman) (20 dəq)

### 5.1 ECDH protokolunun iş prinsipi

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 ECDH açar razılaşdırması</h4>
<ol>
    <li>Alis private scalar $a$ və public key $A=[a]G$ yaradır.</li>
    <li>Bob private scalar $b$ və public key $B=[b]G$ yaradır.</li>
    <li>Alis $Z=[a]B$, Bob isə $Z=[b]A$ hesablayır; düzgün parametrlərdə nəticələr eynidir.</li>
    <li>Raw $Z$ birbaşa tətbiq açarı deyil. KDF həm $Z$-ni, həm protokol/rol kontekstini, həm də public key-ləri bağlayaraq keying material çıxarır.</li>
</ol>
<p><b>Təhlükəsizlik sərhədi:</b> sadə ephemeral ECDH peer-i autentifikasiya etmir və MITM hücumunu təkbaşına dayandırmır. Real protokol sertifikat, PSK və ya imza ilə transcript-i autentifikasiya etməli, peer public key validation və lazım olduqda key confirmation tətbiq etməlidir.</p>
</div>


### 5.2 cryptography ilə ECDH

In [ ]:
def ec_public_bytes(public_key):
    return public_key.public_bytes(
        encoding=serialization.Encoding.X962,
        format=serialization.PublicFormat.UncompressedPoint,
    )


def derive_ecdh_key(shared_secret, alice_public_bytes, bob_public_bytes):
    """Raw ECDH output-dan transcript-bound 32-byte demo key çıxarır."""
    transcript = b"AS22|ECDH|v1|Alice|Bob|" + alice_public_bytes + bob_public_bytes
    return HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=b"AS22-ECDH-demo-salt",
        info=transcript,
    ).derive(shared_secret)


def ecdh_demo(curve=None):
    curve = curve or ec.SECP256R1()
    alice_private = ec.generate_private_key(curve)
    bob_private = ec.generate_private_key(curve)
    alice_public = alice_private.public_key()
    bob_public = bob_private.public_key()

    alice_shared = alice_private.exchange(ec.ECDH(), bob_public)
    bob_shared = bob_private.exchange(ec.ECDH(), alice_public)
    assert alice_shared == bob_shared

    alice_public_bytes = ec_public_bytes(alice_public)
    bob_public_bytes = ec_public_bytes(bob_public)
    alice_key = derive_ecdh_key(alice_shared, alice_public_bytes, bob_public_bytes)
    bob_key = derive_ecdh_key(bob_shared, alice_public_bytes, bob_public_bytes)
    assert alice_key == bob_key and len(alice_key) == 32

    result = {
        "curve": curve.name,
        "raw_shared_length": len(alice_shared),
        "derived_length": len(alice_key),
        "shared_equal": True,
        "derived_equal": True,
    }
    print(f"✅ ECDH {curve.name}: raw={len(alice_shared)} bayt, derived=32 bayt; secrets göstərilmir")
    return result


ecdh_result = ecdh_demo()
assert ecdh_result["shared_equal"] and ecdh_result["derived_equal"]


### 5.3 Müxtəlif əyrilərlə lokal ECDH ölçümü

Bu mikroölçüm yalnız cari Python/OpenSSL/platforma kombinasiyasında **key generation + exchange** median vaxtını göstərir. Bir nəticədən universal sürət və ya enerji üstünlüyü çıxarılmır; funksional bərabərlik hər təkrarda ayrıca yoxlanır.


In [ ]:
def benchmark_ecdh_curve(curve, rounds=7):
    """Silent repeated local measurement; security benchmark deyil."""
    samples_ns = []
    shared_length = None
    for _ in range(rounds):
        start = time.perf_counter_ns()
        alice_private = ec.generate_private_key(curve)
        bob_private = ec.generate_private_key(curve)
        alice_shared = alice_private.exchange(ec.ECDH(), bob_private.public_key())
        bob_shared = bob_private.exchange(ec.ECDH(), alice_private.public_key())
        samples_ns.append(time.perf_counter_ns() - start)
        assert alice_shared == bob_shared
        shared_length = len(alice_shared)
    return {
        "curve": curve.name,
        "rounds": rounds,
        "median_ms": statistics.median(samples_ns) / 1_000_000,
        "raw_shared_length": shared_length,
    }


def ecdh_curve_comparison():
    curves = [ec.SECP256R1(), ec.SECP384R1(), ec.SECP521R1(), ec.SECP256K1()]
    results = [benchmark_ecdh_curve(curve) for curve in curves]
    for result in results:
        print(
            f"{result['curve']:12} raw={result['raw_shared_length']:2} bayt; "
            f"median keygen+exchange={result['median_ms']:.3f} ms ({result['rounds']} raund)"
        )
    assert all(result["rounds"] == 7 for result in results)
    return results


ecdh_benchmarks = ecdh_curve_comparison()


### ✍️ Çalışma 1: ECDH (1 bal)

1. SECP256R1 ilə iki tərəfli ECDH implementasiya edin və raw shared output-ların eyni olduğunu assertion ilə yoxlayın.
2. Hər iki tərəfdə public key-ləri və rol/protokol kontekstini eyni HKDF `info` daxilinə bağlayıb 32-baytlıq key çıxarın.
3. Raw shared secret və derived key-i ekrana çıxarmayın; yalnız uzunluq və bərabərlik göstərin.
4. MITM-dən qorunmaq üçün bu primitive-in hansı autentifikasiya qatına ehtiyacı olduğunu izah edin.
5. Əyrilərin lokal vaxtını bir neçə raundla ölçün; universal “ən sürətli” nəticəsi çıxarmayın.

**Yoxlama meyarı:** iki raw nəticə və iki derived key eynidir; derived key 32 baytdır; benchmark bütün raundlarda round-trip assertion keçir.


## ✍️ ECDSA (Elliptic Curve Digital Signature Algorithm) (25 dəq)

### 6.1 ECDSA-nın iş prinsipi

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔏 ECDSA imza sxemi</h4>
<p>Domen parametrləri $E,G,n$ və private/public key $d,Q=[d]G$ verilsin.</p>
<ol>
    <li>Hər imza üçün yeni, gizli və qorunan $k$, $0&lt;k&lt;n$ yaradılır; FIPS 186-5 həm təsadüfi, həm deterministik generasiyanı müəyyən edir.</li>
    <li>$R=[k]G$, $r=x_R\bmod n$; $r=0$ olarsa yeni $k$ seçilir.</li>
    <li>Hash nəticəsindən $n$-in bit uzunluğuna uyğun $z$ formalaşdırılır.</li>
    <li>$s=k^{-1}(z+rd)\bmod n$; $s=0$ olarsa yeni $k$ seçilir.</li>
    <li>Verifier $r,s$ range-ni, public key/domain parametrlərini və qrup tənliyini yoxlayır.</li>
</ol>
<p><b>Vacib:</b> $k$ reuse və ya $k$ sızması private key-i ifşa edə bilər. Kod custom nonce yaratmır; `cryptography` kitabxanasının public signing API-sindən istifadə edir.</p>
</div>


### 6.2 cryptography ilə ECDSA

In [ ]:
def ecdsa_demo():
    private_key = ec.generate_private_key(ec.SECP256R1())
    public_key = private_key.public_key()
    message = b"AS22 ECDSA test message"
    signature = private_key.sign(message, ec.ECDSA(hashes.SHA256()))

    public_key.verify(signature, message, ec.ECDSA(hashes.SHA256()))

    modified_message_rejected = False
    try:
        public_key.verify(signature, message + b"!", ec.ECDSA(hashes.SHA256()))
    except InvalidSignature:
        modified_message_rejected = True

    tampered_signature = bytearray(signature)
    tampered_signature[-1] ^= 1
    modified_signature_rejected = False
    try:
        public_key.verify(bytes(tampered_signature), message, ec.ECDSA(hashes.SHA256()))
    except InvalidSignature:
        modified_signature_rejected = True

    assert modified_message_rejected and modified_signature_rejected
    result = {
        "curve": private_key.curve.name,
        "signature_length": len(signature),
        "message_tamper_rejected": modified_message_rejected,
        "signature_tamper_rejected": modified_signature_rejected,
    }
    print(f"✅ ECDSA {private_key.curve.name}: verify və iki negative test keçdi")
    return private_key, message, signature, result


ecdsa_private_key, ecdsa_message, ecdsa_signature, ecdsa_result = ecdsa_demo()


### 6.3 ECDSA imza formatları

`cryptography` ECDSA imzasını ASN.1 DER ilə kodlanmış `(r,s)` kimi qaytarır; uzunluq dəyişə bilər. Bəzi protokollar sabit enli `r || s` (“raw”/P1363) tələb edir. Formatlar kontekstdən kənar dəyişdirilməməlidir.


In [ ]:
def ecdsa_signature_formats(private_key, message, signature_der):
    width = (private_key.curve.key_size + 7) // 8
    r, s = decode_dss_signature(signature_der)
    signature_raw = r.to_bytes(width, "big") + s.to_bytes(width, "big")
    assert len(signature_raw) == 2 * width

    recovered_r = int.from_bytes(signature_raw[:width], "big")
    recovered_s = int.from_bytes(signature_raw[width:], "big")
    recovered_der = encode_dss_signature(recovered_r, recovered_s)
    assert recovered_der == signature_der
    private_key.public_key().verify(recovered_der, message, ec.ECDSA(hashes.SHA256()))

    result = {
        "der_length": len(signature_der),
        "raw_length": len(signature_raw),
        "round_trip": True,
    }
    print(f"✅ ECDSA format round-trip: DER={len(signature_der)}, raw={len(signature_raw)} bayt")
    return result


ecdsa_format_result = ecdsa_signature_formats(
    ecdsa_private_key, ecdsa_message, ecdsa_signature
)


### ✍️ Çalışma 2: ECDSA (1 bal)

1. ECDSA/SHA-256 ilə mesajı imzalayın və public key ilə yoxlayın.
2. Mesajın bir baytını və imzanın bir baytını ayrıca dəyişin; hər iki yoxlamanın `InvalidSignature` ilə rədd edildiyini assertion ilə göstərin.
3. DER `(r,s)` imzasını decode edin, P-256 üçün 64-baytlıq `r||s` formasına çevirin və DER-ə geri qaytarıb yoxlayın.
4. ECDSA per-message secret `k` reuse-in niyə private key üçün təhlükəli olduğunu izah edin.

**Yoxlama meyarı:** positive verify keçir; iki negative verify rədd edilir; DER↔raw round-trip ilkin imzaya bərabərdir.


## ⚡ Ed25519 (10 dəq)

### 7.1 Ed25519 xüsusiyyətləri

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>⚡ Ed25519</h4>
<p>Ed25519, RFC 8032-də təsvir olunan EdDSA instansiyasıdır və $p=2^{255}-19$ üzərində edwards25519-dan istifadə edir.</p>
<ul>
    <li><b>Twisted Edwards forması:</b> $-x^2+y^2=1+d x^2y^2$ (RFC parametri $a=-1$).</li>
    <li><b>Public key encoding:</b> 32 bayt; <b>signature:</b> 64 bayt.</li>
    <li><b>Deterministik imza:</b> eyni key/message üçün eyni signature alınır; ayrıca random nonce API-si yoxdur.</li>
    <li><b>Variant sərhədi:</b> Ed25519, Ed25519ctx və Ed25519ph eyni şey deyil; protokol düzgün variantı sabitləməlidir.</li>
    <li><b>Implementasiya sərhədi:</b> constant-time və side-channel resistance sxemin adından avtomatik çıxmır; konkret kitabxana və platforma ayrıca qiymətləndirilir.</li>
</ul>
</div>


In [ ]:
def ed25519_demo():
    private_key = ed25519.Ed25519PrivateKey.generate()
    public_key = private_key.public_key()
    public_bytes = public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )
    message = b"AS22 Ed25519 test message"
    signature = private_key.sign(message)
    repeated_signature = private_key.sign(message)

    assert len(public_bytes) == 32
    assert len(signature) == 64
    assert signature == repeated_signature
    public_key.verify(signature, message)

    tamper_rejected = False
    try:
        public_key.verify(signature, message + b"!")
    except InvalidSignature:
        tamper_rejected = True
    assert tamper_rejected

    result = {
        "public_length": len(public_bytes),
        "signature_length": len(signature),
        "deterministic": True,
        "tamper_rejected": True,
    }
    print("✅ Ed25519: 32-byte public key, 64-byte signature, deterministic və tamper-rejected")
    return result


ed25519_result = ed25519_demo()


### ✍️ Çalışma 3: Ed25519 (1 bal)

1. Ed25519 key pair yaradın, mesajı imzalayın və yoxlayın.
2. Public key və signature encoding uzunluqlarını assertions ilə yoxlayın.
3. Eyni key/message-i iki dəfə imzalayıb deterministik nəticəni göstərin; dəyişmiş mesajı rədd edin.
4. Ed25519 və ECDSA-nı nonce modeli, encoding, protocol compatibility və implementation assurance baxımından müqayisə edin; “biri həmişə üstündür” nəticəsi çıxarmayın.

**Yoxlama meyarı:** 32-bayt public key, 64-bayt signature, eyni imzalar və uğurlu negative test.


## 🏛️ NIST əyriləri və Dual_EC_DRBG ayrımı (10 dəq)

### 8.1 NIST əyrilərinin xüsusiyyətləri

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🏛️ NIST prime-field Weierstrass əyriləri</h4>
<p>NIST SP 800-186 P-256, P-384 və P-521 domen parametrlərini müəyyən edir. NIST SP 800-57 ilə uyğun təxmini klassik təhlükəsizlik gücləri:</p>
<ul>
    <li><b>P-256 / secp256r1:</b> təxminən 128 bit; $p=2^{256}-2^{224}+2^{192}+2^{96}-1$.</li>
    <li><b>P-384 / secp384r1:</b> təxminən 192 bit; $p=2^{384}-2^{128}-2^{96}+2^{32}-1$.</li>
    <li><b>P-521 / secp521r1:</b> təxminən 256 bit; $p=2^{521}-1$.</li>
</ul>
<p>RFC 8446 bu üç qrupu TLS 1.3 `supported_groups` siyahısında müəyyən edir; konkret deployment dəstəyi kitabxana və konfiqurasiyadan asılıdır.</p>
</div>


### 8.2 NIST əyriləri və Dual_EC_DRBG — düzgün ayrım

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<p>2013-cü ildə Dual_EC_DRBG random-bit generatorunun xüsusi $P,Q$ nöqtələri ilə bağlı ictimai backdoor narahatlığı gücləndi. NIST 2014-cü ildə onu çıxarmaq qərarını açıqladı və SP 800-90A Rev. 1 (2015) Dual_EC_DRBG-ni tövsiyədən çıxardı.</p>
<p><b>Bu hadisə P-256/P-384/P-521 əyrilərində backdoor olduğunu sübut etmir.</b> Dual_EC_DRBG ayrıca DRBG konstruksiyası və əlavə nöqtə parametrləri idi; standart əyrilər isə ECDH/ECDSA üçün ayrıca domen parametrləridir. Etimad və parameter rigidity müzakirə oluna bilər, amma faktla ehtimal ayrılmalıdır.</p>
</div>


### ✍️ Çalışma 4: NIST əyriləri (1 bal)

1. P-256, P-384 və P-521 ilə ECDH round-trip aparın.
2. NIST SP 800-57 təhlükəsizlik güclərini və raw shared-output uzunluqlarını fərqli anlayışlar kimi müqayisə edin.
3. Dual_EC_DRBG-nin hansı konstruktor/parametrlərlə bağlı olduğunu və NIST-in 2014/2015 qərarını izah edin.
4. Niyə Dual_EC hadisəsi P-256/P-384/P-521-də backdoor-un sübutu deyil?

**Yoxlama meyarı:** hər curve üçün iki tərəfin raw output-u eynidir; tarixi cavab NIST-in rəsmi qərarına istinad edir və DRBG ilə curve domain parameters-i ayırır.


## 🔄 Curve25519 və X25519 (10 dəq)

### 9.1 Curve25519 və X25519

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<ul>
    <li><b>Curve25519:</b> $p=2^{255}-19$ üzərində Montgomery əyrisi $v^2=u^3+486662u^2+u$.</li>
    <li><b>X25519:</b> RFC 7748-də təyin olunan 32-bayt scalar/public u-coordinate funksiyası; key agreement primitive-dir, signature deyil.</li>
    <li><b>KDF:</b> raw X25519 output, hər iki public key və protocol/role context KDF-ə daxil edilir.</li>
    <li><b>Small-order input:</b> all-zero shared output aşkarlanarsa protokol abort etməlidir; istifadə etdiyimiz `cryptography` API-si null shared key-i xəta ilə rədd edir və bu davranış test olunur.</li>
    <li><b>Implementasiya:</b> Montgomery ladder regular struktur verir, lakin konkret constant-time/side-channel zəmanəti kitabxana və platformadan asılıdır.</li>
    <li><b>Protokollar:</b> RFC 8446 X25519-u TLS 1.3 named group kimi müəyyən edir; WireGuard protokolu da Curve25519 scalar multiplication istifadə edir.</li>
</ul>
</div>


In [ ]:
def x25519_public_bytes(public_key):
    return public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )


def derive_x25519_key(shared_secret, alice_public_bytes, bob_public_bytes):
    transcript = b"AS22|X25519|v1|Alice|Bob|" + alice_public_bytes + bob_public_bytes
    return HKDF(
        algorithm=hashes.SHA256(),
        length=32,
        salt=b"AS22-X25519-demo-salt",
        info=transcript,
    ).derive(shared_secret)


def x25519_demo():
    alice_private = x25519.X25519PrivateKey.generate()
    bob_private = x25519.X25519PrivateKey.generate()
    alice_public = alice_private.public_key()
    bob_public = bob_private.public_key()

    alice_shared = alice_private.exchange(bob_public)
    bob_shared = bob_private.exchange(alice_public)
    assert alice_shared == bob_shared and alice_shared != bytes(32)

    alice_public_bytes = x25519_public_bytes(alice_public)
    bob_public_bytes = x25519_public_bytes(bob_public)
    alice_key = derive_x25519_key(alice_shared, alice_public_bytes, bob_public_bytes)
    bob_key = derive_x25519_key(bob_shared, alice_public_bytes, bob_public_bytes)
    assert alice_key == bob_key and len(alice_key) == 32

    null_rejected = False
    try:
        null_public = x25519.X25519PublicKey.from_public_bytes(bytes(32))
        alice_private.exchange(null_public)
    except ValueError:
        null_rejected = True
    assert null_rejected

    result = {
        "public_length": len(alice_public_bytes),
        "raw_shared_length": len(alice_shared),
        "derived_length": len(alice_key),
        "null_shared_rejected": True,
    }
    print("✅ X25519: 32-byte encodings, HKDF agreement və null-shared negative test keçdi")
    return result


x25519_result = x25519_demo()


### 9.2 secp256k1 (Bitcoin əyrisi)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<ul>
    <li><b>Əyri:</b> $y^2=x^3+7$ over $\mathbb F_p$, $p=2^{256}-2^{32}-977$.</li>
    <li><b>Baza nöqtəsinin sırası:</b> təxminən 256 bit; klassik təhlükəsizlik gücü təxminən 128 bit.</li>
    <li><b>İstifadə:</b> Bitcoin imzaları secp256k1 qrupundan istifadə edir; tətbiq olunan signature qaydaları protokol səviyyəsində ayrıca müəyyən olunur.</li>
    <li><b>Parametrlər:</b> $a=0,b=7$ kimi sadə əmsallar açıqdır, lakin bu təkbaşına implementasiya təhlükəsizliyi və ya bütün protokollardan üstünlük sübut etmir.</li>
</ul>
</div>


### ✍️ Çalışma 5: X25519 və secp256k1 (1 bal)

1. X25519 ilə iki tərəfli exchange və transcript-bound HKDF implementasiya edin.
2. All-zero public input-un rədd edildiyini negative test ilə göstərin.
3. secp256k1 ilə ECDSA/SHA-256 imzası yaradın və dəyişmiş mesajı rədd edin.
4. X25519 və P-256 ECDH-ni encoding, validation, protocol support və implementation assurance baxımından müqayisə edin; yalnız lokal benchmark-a görə universal seçim etməyin.

**Yoxlama meyarı:** X25519 key agreement/KDF uyğun gəlir, null-shared test rədd edilir, secp256k1 signature positive/negative verify keçir.


## 🧠 Brainpool əyriləri (5 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<p>RFC 5639 Brainpool prime-field domen parametrlərini və onların sistematik, təkrar istehsal edilə bilən, verifiably pseudo-random generasiya prosesini təsvir edir.</p>
<ul>
    <li>160–512 bit field ölçüləri üçün parametr dəstləri verilir.</li>
    <li>RFC-dəki əsas qrupların kofaktoru 1-dir və qrup sırası sadədir.</li>
    <li>RFC 5639 informational sənəddir; konkret protokolda support və uyğun profil ayrıca yoxlanmalıdır.</li>
    <li>Şəffaf generasiya prosesi faydalı assurance-dır, amma “heç bir backdoor mümkün deyil” sübutu deyil.</li>
</ul>
</div>


## 🛡️ SafeCurves qiymətləndirmə meyarları (5 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<p>SafeCurves (Bernstein–Lange) müxtəlif curve-ləri bir neçə açıq meyar üzrə müqayisə edən tədqiqat layihəsidir; normativ standart və ya bütün protokollar üçün universal “recommended list” deyil.</p>
<ul>
    <li><b>ECDLP meyarları:</b> field, equation/base point, rho, transfer və discriminant hücumları.</li>
    <li><b>Rigidity:</b> parametrlərin seçim prosesində manipulyasiya imkanının qiymətləndirilməsi.</li>
    <li><b>Implementation meyarları:</b> ladder, twist security, complete formulas və point-encoding indistinguishability.</li>
    <li><b>Kofaktor sərhədi:</b> “twist-safe” olmaq “kiçik altqrup yoxdur” demək deyil; məsələn Curve25519-un kofaktoru 8-dir və protocol/API input handling qaydaları vacibdir.</li>
</ul>
</div>


## 🖥️ İnteqrasiya edilmiş özünü-yoxlama (15 dəq)

Bu bölmə əvvəlki bütün protocol nümunələrini qeyri-interaktiv şəkildə birləşdirir. Hər branch məcburidir; regression olduqda assertion notebook-u dayandırır.


In [ ]:
def ecc_protocols_self_test():
    assert ecdh_result["shared_equal"] and ecdh_result["derived_length"] == 32
    assert len(ecdh_benchmarks) == 4
    assert ecdsa_result["message_tamper_rejected"]
    assert ecdsa_result["signature_tamper_rejected"]
    assert ecdsa_format_result["round_trip"]
    assert ed25519_result == {
        "public_length": 32,
        "signature_length": 64,
        "deterministic": True,
        "tamper_rejected": True,
    }
    assert x25519_result["null_shared_rejected"]

    secp_private = ec.generate_private_key(ec.SECP256K1())
    message = b"AS22 secp256k1 signature test"
    signature = secp_private.sign(message, ec.ECDSA(hashes.SHA256()))
    secp_private.public_key().verify(signature, message, ec.ECDSA(hashes.SHA256()))
    rejected = False
    try:
        secp_private.public_key().verify(signature, message + b"!", ec.ECDSA(hashes.SHA256()))
    except InvalidSignature:
        rejected = True
    assert rejected

    print("✅ ECC protocol self-test: ECDH/ECDSA/Ed25519/X25519/secp256k1 keçdi")
    return True


assert ecc_protocols_self_test()


## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: ECC protokolları paketi (3 bal)

```
ecc_protocols/
├── __init__.py
├── ecdh.py          # ECDH + transcript-bound KDF; raw secret log edilmir
├── ecdsa.py         # DER/raw format və positive/negative verification
├── ed25519.py       # Ed25519 sign/verify və encoding assertions
├── x25519.py        # X25519 + KDF + null-shared rejection
└── tests/           # avtomatik testlər; interaktiv input yoxdur
```

**Tələblər:** public API-lərdən istifadə edin; private/shared/derived key-ləri log etməyin; protocol context-i KDF-ə bağlayın; yalnız `InvalidSignature` kimi gözlənən xətaları tutun; hər modul üçün positive və negative test yazın.


### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

1. ECDH raw output-dan public-key/role context-li HKDF vasitəsilə iki eyni key çıxarın.
2. ECDSA ilə byte fayl məzmununu imzalayın; dəyişmiş content və signature-ı rədd edin.
3. Ed25519 üçün public/signature encoding uzunluqlarını və deterministik imzanı yoxlayın.
4. X25519 üçün key agreement, transcript-bound KDF və null-shared negative test yazın.
5. Lokal ECDH ölçümünü ən azı 7 raundla aparın, medianı göstərin və nəticəni platforma/versiya ilə qeyd edin.


### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

1. **Ed25519/X25519 ayrımı:** RFC 8032 və RFC 7748 əsasında curve, signature scheme və key-agreement function anlayışlarını ayırın.
2. **Dual_EC_DRBG:** NIST-in 2014 qərarı və SP 800-90A Rev. 1 əsasında DRBG hadisəsini P-curve domen parametrlərindən ayırın; sübut olunmuş faktı ehtimaldan fərqləndirin.
3. **secp256k1:** SEC 2 və Bitcoin protokol sənədləri əsasında curve parametrlərini və protocol-level signature qaydalarını izah edin; “niyə seçildi” barədə mənbəsiz motiv yazmayın.
4. **SafeCurves:** layihənin normativ standart olmadığını qeyd edib rigidity, ladder, twist və completeness meyarlarını müqayisə edin.

**Format:** 1–2 səhifə PDF; ən azı 5 mənbə; RFC/NIST/SEC kimi primary mənbələrə üstünlük; mənbə tarixi və versiyası göstərilir.


## 📌 Yekun

* **ECDH:** raw group output yalnız KDF və autentifikasiya olunmuş protocol context ilə keying material-a çevrilir.
* **ECDSA:** hər imza üçün qorunan per-message secret lazımdır; DER və raw encoding fərqlidir.
* **Ed25519:** RFC 8032 signature scheme-dir; 32-bayt public key, 64-bayt signature və deterministik signing davranışı yoxlanıldı.
* **Curve25519/X25519:** curve ilə key-agreement function eyni termin deyil; all-zero shared output rədd edilir.
* **NIST/Dual_EC:** Dual_EC_DRBG ayrıca çıxarılmış random generator idi və P-curves-da backdoor-un sübutu deyil.
* **Curve seçimi:** security strength, rigidity, protocol support və implementation assurance birlikdə qiymətləndirilir; tək benchmark və ya curve adı kifayət deyil.

### 💭 Müzakirə sualları

1. Niyə raw ECDH/X25519 output birbaşa tətbiq açarı deyil?
2. ECDSA nonce reuse ilə Ed25519 deterministik signing arasında nə fərq var?
3. Dual_EC_DRBG hadisəsini P-256 domen parametrlərinə avtomatik aid etmək niyə səhvdir?
4. SafeCurves meyarı ilə konkret kitabxananın side-channel assurance-ı necə fərqlənir?


## 📚 Əlavə resurslar

Əsas və normativ mənbələr:

* NIST SP 800-56A Rev. 3 — elliptik-əyri key establishment: https://doi.org/10.6028/NIST.SP.800-56Ar3
* NIST SP 800-186 — elliptik-əyri domen parametrləri: https://doi.org/10.6028/NIST.SP.800-186
* FIPS 186-5 — ECDSA və EdDSA: https://doi.org/10.6028/NIST.FIPS.186-5
* RFC 7748 — X25519/X448: https://www.rfc-editor.org/rfc/rfc7748
* RFC 8032 — Ed25519/Ed448: https://www.rfc-editor.org/rfc/rfc8032
* RFC 8446 — TLS 1.3 supported groups: https://www.rfc-editor.org/rfc/rfc8446
* RFC 5639 — Brainpool parameters/generation: https://www.rfc-editor.org/rfc/rfc5639
* NIST (2014), Dual_EC_DRBG removal decision: https://www.nist.gov/news-events/news/2014/04/nist-removes-cryptography-algorithm-random-number-generator-recommendations
* SafeCurves evaluation project: https://safecurves.cr.yp.to/

> Kod tədris üçündür. Real protokolda named profile, autentifikasiya, key lifecycle və platformaya uyğun yoxlanılmış kitabxana tələb olunur.

---

✅ **Məşğələ 22 tamamlandı:** bütün məcburi code cells yuxarıdan aşağıya avtomatik yoxlanır.
